In [0]:
sales_df = spark.read.format("csv")\
    .option("Header","true")\
    .option("inferSchema","true")\
    .load("/Volumes/sql_problems/default/my_volume/day08_sales.csv")

display(sales_df)

In [0]:
from pyspark.sql.functions import col, sum

# Pivot the quarter column into wide-format columns
pivoted_df = sales_df.groupBy("store_id", "category") \
    .pivot("quarter", ["Q1", "Q2", "Q3"]) \
    .sum("sales") \
    .na.fill(0) 
    # Optional: replaces nulls with 0 for missing quarters

display(pivoted_df)

In [0]:
sales_df.createOrReplaceTempView("sales")

In [0]:
%sql
SELECT store_id,category, Q1, Q2, Q3
FROM (
    SELECT store_id, category, quarter, sales
    FROM sales
)
PIVOT (
    SUM(sales)
    FOR quarter IN ('Q1', 'Q2', 'Q3')
)
ORDER BY store_id, category;

In [0]:
%sql
SELECT 
    store_id,
    category,
    SUM(CASE WHEN quarter = 'Q1' THEN sales ELSE 0 END) AS Q1,
    SUM(CASE WHEN quarter = 'Q2' THEN sales ELSE 0 END) AS Q2,
    SUM(CASE WHEN quarter = 'Q3' THEN sales ELSE 0 END) AS Q3
FROM sales
GROUP BY store_id, category
ORDER BY store_id, category;